In [1]:
! pip install datasets evaluate

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Users\pd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from datasets import load_dataset
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# TODO: Import other classifiers you want to test
# Added:
# - LinearSVC as a strong SVM baseline for text classification
# - MultinomialNB as an additional legacy ML baseline

# TODO: Load a text classification dataset
dataset = load_dataset("imdb")

# TODO: Convert the dataset splits into pandas DataFrames
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Optional: inspect class balance to answer whether IMDb is balanced
print("Train label counts:")
print(train_df["label"].value_counts().sort_index())

print("\nTest label counts:")
print(test_df["label"].value_counts().sort_index())

print("\nTrain label percentages:")
print((train_df["label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTest label percentages:")
print((test_df["label"].value_counts(normalize=True).sort_index() * 100).round(2))

# TODO: Assign the relevant dataset columns to feature and label variables
X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

# TODO: Transform the text data into numerical features using TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2)
)

# TODO: Convert the transformed data into a format suitable for classifiers
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# TODO: Define a set of classifiers to evaluate
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Multinomial Naive Bayes": MultinomialNB()
}

# TODO: Loop through classifiers, train them, and evaluate their performance

# OPTIONAL: You can use lazypredict, but you may run out of memory

for model_name, model in classifiers.items():
    try:
        print(f"\n--- Running: {model_name} ---")

        # TODO: Train the model using training data
        model.fit(X_train_tfidf, y_train)

        # TODO: Generate predictions on test data
        y_pred = model.predict(X_test_tfidf)

        # TODO: Generate and print a classification report (scroll all the way down for some sample code)
        print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

        # Extra: print a single F1 score for easier comparison
        f1 = f1_score(y_test, y_pred, average="binary")
        print(f"{model_name} F1 Score: {f1:.4f}")

        # TODO, IF NEEDED: Clean up memory after each classifier runs

    except Exception as e:
        print(f"Error with {model_name}: {e}")
        continue  # Skip to the next classifier if an error occurs

Train label counts:
label
0    12500
1    12500
Name: count, dtype: int64

Test label counts:
label
0    12500
1    12500
Name: count, dtype: int64

Train label percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

Test label percentages:
label
0    50.0
1    50.0
Name: proportion, dtype: float64

--- Running: Logistic Regression ---
              precision    recall  f1-score   support

    Negative       0.88      0.88      0.88     12500
    Positive       0.88      0.89      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000

Logistic Regression F1 Score: 0.8830

--- Running: Linear SVM ---
              precision    recall  f1-score   support

    Negative       0.86      0.88      0.87     12500
    Positive       0.88      0.86      0.87     12500

    accuracy                           0.87     25000
   macro avg       0.87      0.87 

In [18]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


dataset = load_dataset("imdb")


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(["text"]).rename_column("label", "labels")

tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"]

model = AutoModelForSequenceClassification.from_pretrained("bert-large-uncased", num_labels=2)

# TODO: PLAY WITH THE HYPERPARAMETERS
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.1,
    logging_dir="./logs",
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
results = trainer.evaluate(eval_dataset)
print("Evaluation Results:", results)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [20]:
# TODO: Adapt to Sentence BERT

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# Load the IMDb dataset
dataset = load_dataset("imdb")

train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

# Load Sentence-BERT model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Convert text reviews into dense embeddings
X_train_embeddings = model.encode(train_texts, batch_size=32, show_progress_bar=True)
X_test_embeddings = model.encode(test_texts, batch_size=32, show_progress_bar=True)

# Train a simple classifier on top of the embeddings
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_embeddings, train_labels)

# Predict on the test set
y_pred = classifier.predict(X_test_embeddings)

# Evaluate performance
print(classification_report(test_labels, y_pred, target_names=["Negative", "Positive"]))

f1 = f1_score(test_labels, y_pred, average="binary")
print(f"Sentence-BERT F1 Score: {f1:.4f}")

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    Negative       0.81      0.81      0.81     12500
    Positive       0.81      0.81      0.81     12500

    accuracy                           0.81     25000
   macro avg       0.81      0.81      0.81     25000
weighted avg       0.81      0.81      0.81     25000

Sentence-BERT F1 Score: 0.8103


In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Generate predictions with the fine-tuned BERT model
predictions = trainer.predict(eval_dataset)
logits = predictions.predictions
y_pred = np.argmax(logits, axis=1)

# Retrieve true labels safely
if "labels" in eval_dataset.column_names:
    y_true = eval_dataset["labels"]
elif "label" in eval_dataset.column_names:
    y_true = eval_dataset["label"]
else:
    raise ValueError("Neither 'labels' nor 'label' found in dataset!")

# Print detailed classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=["Negative", "Positive"]
)
print("Classification Report:")
print(report)

# Print a single F1 score for easier interpretation
f1 = f1_score(y_true, y_pred, average="binary")
print(f"BERT F1 Score: {f1:.4f}")